# 공개 COVID-19 코호트 재현 (ver8, Supp. Fig. 3) 검산
2026-11-18. ① 원본 `README_Public_Replication_ver8.md`의 수치를 `data/*.csv`와 대조하고(FAIL = 원본 README 오류), ② 파이프라인과 별개로 다시 구현한 `verify_public_independent.py`를 실행합니다.
`PUB_DATA` = 16번 폴더의 `data/`, `PBISC_ROOT` = `public_covid_who_hspc_nk_20261115/`와 `HSPC_CD56brightNK_COVID_analysis_summary_20261116/`가 있는 폴더.

In [ ]:
import os, numpy as np, pandas as pd
D = os.environ.get('PUB_DATA', 'data'); rd = lambda f: pd.read_csv(os.path.join(D, f))
C, M, H = rd('public_cohort_level_results_v8.csv'), rd('public_meta_analysis_v8.csv').set_index('feature'), rd('public_hspc_timewindow_death_v8.csv')
c = C.set_index(['cohort','feature']); issues = []
def check(label, ok, detail=''):
    print(('PASS ' if ok else 'FAIL ') + label + ('' if ok else f'  -> {detail}'))
    if not ok: issues.append(label)
r2 = lambda x: round(float(x), 2)

## 1. 코호트 크기

In [ ]:
n = C.groupby('cohort').n_sev.max(); d = C.groupby('cohort').n_death_all.max(); print(pd.DataFrame({'n_WHO_graded': n, 'deaths': d}))
check('README: total N = 222', n.sum()==222, f'N = {n.sum()}')
check('README: Liu n = 32', n['Liu 2021']==32, f"Liu n = {n['Liu 2021']}")
check('README: 23 deaths', d.sum()==23)
check('Stephenson 73 / COMBAT 75 / vdW 42', [n['Stephenson 2021'], n['COMBAT 2022'], n['van der Wijst 2021']]==[73,75,42])

## 2. 메타분석 (표 1–4행, 제외 대조)

In [ ]:
def meta(f, rho, p, k=None, N=None, rtol=0.06):
    r = M.loc[f]; ok = r2(r.rho_meta)==rho and np.isclose(r.p_rho_meta, p, rtol=rtol) and (k is None or r.k_cohorts==k) and (N is None or r.N_patients==N)
    check(f'meta {f}: rho {rho:+.2f}, P {p:g}' + (f', k={k}, N={N}' if k else ''), ok, f'rho {r.rho_meta:+.3f}, P {r.p_rho_meta:.2g}, k {r.k_cohorts}, N {r.N_patients}')
meta('NK_CD56bright', -0.29, 1.0e-4, 3, 190); meta('B_ASC_dividing', 0.33, 5.2e-4, 2, 115); meta('T_CD8_CM', -0.33, 0.0026, 2, 148)
meta('CD16_mono_ofMono', -0.35, 0.0093, 4, 219); meta('B_ASC_total', 0.33, 5.6e-5, None, 190); meta('T_CD8_CTL', -0.19, 0.052)
meta('HSPC', 0.45, 2.8e-8); meta('NK_CD56dim', -0.06, 0.39)
check('HSPC I2 21%, CTL I2 43%', round(M.loc['HSPC','I2_rho'])==21 and round(M.loc['T_CD8_CTL','I2_rho'])==43)
check('B ASC total death AUC 0.59 P 0.22; WHO>=6 0.49 P 0.84', (r2(M.loc['B_ASC_total','AUC_death_all_meta']), r2(M.loc['B_ASC_total','p_death_all_meta']), r2(M.loc['B_ASC_total','AUC_death_WHO6_meta']), r2(M.loc['B_ASC_total','p_death_WHO6_meta']))==(0.59,0.22,0.49,0.84))
check('T CD8 CM death AUC 0.31 P 0.038; WHO>=6 0.42 P 0.35', (r2(M.loc['T_CD8_CM','AUC_death_all_meta']), round(M.loc['T_CD8_CM','p_death_all_meta'],3), r2(M.loc['T_CD8_CM','AUC_death_WHO6_meta']), r2(M.loc['T_CD8_CM','p_death_WHO6_meta']))==(0.31,0.038,0.42,0.35))
check('NK CD56dim death AUC 0.50 P 1.00', (r2(M.loc['NK_CD56dim','AUC_death_all_meta']), r2(M.loc['NK_CD56dim','p_death_all_meta']))==(0.5,1.0))

## 3. 코호트별 수치

In [ ]:
S, CO, V = 'Stephenson 2021', 'COMBAT 2022', 'van der Wijst 2021'
check('Stephenson IgG OR 1.95 P 0.0041; IgA OR 1.84 P 0.0025', (r2(c.loc[(S,'B_ASC_IgG'),'OR_glm']), round(c.loc[(S,'B_ASC_IgG'),'p_glm'],4), r2(c.loc[(S,'B_ASC_IgA'),'OR_glm']), round(c.loc[(S,'B_ASC_IgA'),'p_glm'],4))==(1.95,0.0041,1.84,0.0025))
check('README: Stephenson IgG "% B rho +0.26, P 0.024"', False, f"no '% of B' output; IgG Spearman (% PBMC) rho {c.loc[(S,'B_ASC_IgG'),'rho']:+.3f} P {c.loc[(S,'B_ASC_IgG'),'p_sev']:.3f}; B ASC total rho {c.loc[(S,'B_ASC_total'),'rho']:+.3f} P {c.loc[(S,'B_ASC_total'),'p_sev']:.3f}")
check('README: Stephenson T CD8 CM death AUC 0.21-0.26, P<=0.05', False, f"AUC {c.loc[(S,'T_CD8_CM'),'AUC_death_all']:.3f}, P {c.loc[(S,'T_CD8_CM'),'p_death_all']:.3f} (single value)")
check('COMBAT CTL rho -0.31 P 0.0067 adj 0.009', (r2(c.loc[(CO,'T_CD8_CTL'),'rho']), round(c.loc[(CO,'T_CD8_CTL'),'p_sev'],4), round(c.loc[(CO,'T_CD8_CTL'),'p_sev_adj'],3))==(-0.31,0.0067,0.009))
r = c.loc[(V,'T_CD8_CTL')]
check('README: vdW CTL rho -0.36 P 0.022', (r2(r.rho), round(r.p_sev,3))==(-0.36,0.022), f'rho {r.rho:+.3f}, P {r.p_sev:.3f}')
check('README: vdW CTL death AUC 0.26 P 0.049', (r2(r.AUC_death_all), round(r.p_death_all,3))==(0.26,0.049), f'AUC {r.AUC_death_all:.3f}, P {r.p_death_all:.3f}')
check('README: Stephenson CTL rho +0.05', r2(c.loc[(S,'T_CD8_CTL'),'rho'])==0.05, f"rho {c.loc[(S,'T_CD8_CTL'),'rho']:+.3f}")
check('CD16 mono C1 Stephenson rho +0.07; death AUC 0.49', (r2(c.loc[(S,'CD16_mono_C1'),'rho']), r2(c.loc[(S,'CD16_mono_C1'),'AUC_death_all']))==(0.07,0.49))

## 4. HSPC 사망 시간창 (그림 d)

In [ ]:
h = H.set_index(['cohort','window']); F, L7 = 'First acute sample', 'Samples > day 7'; Mt = 'Public Meta (3 cohorts)'
for (co, w), (a, p) in {(S,F):(0.75,0.043),(V,F):(0.78,0.024),(Mt,F):(0.69,0.0072),(V,L7):(0.81,0.032),(Mt,L7):(0.71,0.010)}.items():
    rr = h.loc[(co,w)]; check(f'{co} {w}: AUC {a} P {p}', r2(rr.AUC_all)==a and np.isclose(rr.p_all,p,rtol=0.06), (rr.AUC_all, rr.p_all))
check('Stephenson >7d AUC 0.79', r2(h.loc[(S,L7),'AUC_all'])==0.79)
check('WHO>=6 first-sample meta 0.51 P 0.88', (r2(h.loc[(Mt,F),'AUC_WHO6']), r2(h.loc[(Mt,F),'p_WHO6']))==(0.51,0.88))
check('WHO>=6 Stephenson 0.59->0.65, vdW 0.64->0.71; 4-5 deaths', [r2(h.loc[(S,F),'AUC_WHO6']), r2(h.loc[(S,L7),'AUC_WHO6']), r2(h.loc[(V,F),'AUC_WHO6']), r2(h.loc[(V,L7),'AUC_WHO6'])]==[0.59,0.65,0.64,0.71] and [h.loc[(S,L7),'deaths'], h.loc[(V,L7),'deaths']]==[4,5])

## 5. 메타분석 코호트 구성 민감도: Liu 2021 포함

In [ ]:
from scipy.stats import norm
def dl(q):
    z = np.arctanh(q.rho.values); v = 1.06/(q.n_sev.values-3); w = 1/v; fe = (w*z).sum()/w.sum(); Q = (w*(z-fe)**2).sum(); k = len(z)
    t2 = max(0, (Q-(k-1))/(w.sum()-(w**2).sum()/w.sum())); ws = 1/(v+t2); e = (ws*z).sum()/ws.sum(); s = np.sqrt(1/ws.sum())
    return dict(k=k, rho=np.tanh(e), lo=np.tanh(e-1.96*s), hi=np.tanh(e+1.96*s), P=2*norm.sf(abs(e/s)), I2=max(0,(Q-(k-1))/Q)*100 if Q>0 else 0)
for f in ['NK_CD56bright','B_ASC_total']:
    q = C[C.feature==f]; print(f, 'Liu rho', round(q[q.cohort=='Liu 2021'].rho.iloc[0],3)); print('  without Liu', {k: round(v,4) for k,v in dl(q[q.cohort!='Liu 2021']).items()}); print('  with Liu   ', {k: round(v,4) for k,v in dl(q).items()})

## 6. 독립 재구현 (원본 obs parquet부터)

In [ ]:
import runpy
_ = runpy.run_path(os.environ.get('PUB_INDEP', 'verify_public_independent.py'), run_name='__main__')

## 요약

In [ ]:
print(f'{len(issues)} 불일치 (원본 README 기준)')
for l in issues: print('-', l)